# Bayyinah – Burnout Prediction: Classification Model (`burnout_level`)

**Goal:** predict an employee's burnout level (**Low < Moderate < High < Severe**) from information an HR system can have.

## 1. Setup and loading

In [ ]:
import os, json, time, glob, shutil, warnings, datetime
import numpy as np
import pandas as pd
import sklearn
import joblib
import matplotlib
import matplotlib.pyplot as plt

from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate, train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, cohen_kappa_score)
from sklearn.inspection import permutation_importance
from sklearn.exceptions import InconsistentVersionWarning


try:
    import xgboost
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("XGBoost installed:", HAS_XGB, "| SHAP installed:", HAS_SHAP)

In [ ]:
# ---------------- constants ----------------
RANDOM_STATE = 42
PREP_DIR = "prepared"
MODELS_DIR = "models"

CLASS_ORDER = ["Low", "Moderate", "High", "Severe"]          # ordinal order, used everywhere
LABEL_TO_INT = {c: i for i, c in enumerate(CLASS_ORDER)}     # Low 0, Moderate 1, High 2, Severe 3
INT_TO_LABEL = {i: c for c, i in LABEL_TO_INT.items()}

ID_COL = "employee_id"
LEAKAGE_COLS = ["stress_score", "phq9_score", "phq9_category", "gad7_score", "gad7_category",
                "seeks_mental_health_support", "job_change_intention"]
TARGET_REG, TARGET_CLF = "burnout_score", "burnout_level"

N_ITER = 20              # random search iterations per model and feature list
SUBSAMPLE_FRAC = 0.30    # only for the slow tree models
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


USE_DRIVE = False
DRIVE_PREP_DIR = "/content/drive/MyDrive/Bayyinah/prepared"

REQUIRED_FILES = ["X_train.csv", "X_test.csv", "y_train_clf.csv", "y_test_clf.csv",
                  "y_train_reg.csv", "y_test_reg.csv",
                  "preprocessor_full.pkl", "preprocessor_hr.pkl", "feature_lists.json"]

In [ ]:
def missing_files(folder):
    return [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(folder, f))]

if missing_files(PREP_DIR):
    try:
        from google.colab import drive, files
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    if not IN_COLAB:
        raise FileNotFoundError(f"Missing in '{PREP_DIR}/': {missing_files(PREP_DIR)}")

    if USE_DRIVE:
        drive.mount("/content/drive")
        PREP_DIR = DRIVE_PREP_DIR
    else:
        print("Please upload the prepared zip, or the 9 prepared files.")
        uploaded = files.upload()
        os.makedirs(PREP_DIR, exist_ok=True)
        for name in uploaded:
            if name.endswith(".zip"):
                shutil.unpack_archive(name, "unzipped_tmp")
                for path in glob.glob("unzipped_tmp/**/*", recursive=True):
                    if os.path.basename(path) in REQUIRED_FILES:
                        shutil.copy(path, os.path.join(PREP_DIR, os.path.basename(path)))
            elif name in REQUIRED_FILES:
                shutil.move(name, os.path.join(PREP_DIR, name))

assert not missing_files(PREP_DIR), f"Still missing: {missing_files(PREP_DIR)}"
print("All prepared files found in:", PREP_DIR)

In [ ]:
with open(os.path.join(PREP_DIR, "feature_lists.json")) as f:
    FL = json.load(f)

FEATURES_FULL = FL["FEATURES_FULL"]
FEATURES_HR = FL["FEATURES_HR"]
FEATURE_LISTS = {"FULL": FEATURES_FULL, "HR": FEATURES_HR}
assert FL["burnout_level_classes"] == CLASS_ORDER

X_train = pd.read_csv(os.path.join(PREP_DIR, "X_train.csv"))
X_test = pd.read_csv(os.path.join(PREP_DIR, "X_test.csv"))
y_train = pd.read_csv(os.path.join(PREP_DIR, "y_train_clf.csv"))[TARGET_CLF]
y_test = pd.read_csv(os.path.join(PREP_DIR, "y_test_clf.csv"))[TARGET_CLF]

print("X_train", X_train.shape, "| X_test", X_test.shape)

**Library versions.** A `.pkl` file only loads safely with the same scikit-learn version it was saved with. The loader below turns a version warning into an error. If that happens (or the file doesn't load at all), it **rebuilds the preprocessor from `feature_lists.json` with exactly the same steps** as the preparation notebook (median imputer + StandardScaler, most-frequent imputer + OrdinalEncoder for `company_size`, most-frequent imputer + OneHotEncoder for the rest). Either way we `clone` it, so it is **unfitted** and every pipeline fits its own copy.

In [ ]:
print("pandas       :", pd.__version__)
print("scikit-learn :", sklearn.__version__)
print("numpy        :", np.__version__)
print("joblib       :", joblib.__version__)

In [ ]:
def build_preprocessor(feature_list):
    # same steps as the preparation notebook (used only as a fallback)
    num = [c for c in feature_list if c in FL["numeric"]]
    ordi = [c for c in feature_list if c in FL["categorical_ordinal"]]
    cat = [c for c in feature_list if c in FL["categorical_onehot"]]
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                             ("scaler", StandardScaler())])
    ordinal_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                             ("encoder", OrdinalEncoder(categories=[FL["company_size_order"]]))])
    onehot_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", numeric_pipe, num),
                              ("company_size", ordinal_pipe, ordi),
                              ("cat", onehot_pipe, cat)], remainder="drop")

def load_preprocessor(path, feature_list):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", InconsistentVersionWarning)
            pre = joblib.load(path)
        source = "loaded from .pkl"
    except Exception as e:
        print(f"  ! could not load {path} safely ({type(e).__name__}). Rebuilding from feature_lists.json.")
        pre = build_preprocessor(feature_list)
        source = "rebuilt from feature_lists.json"
    # the preprocessor must use exactly the columns of its feature list
    used = [c for _, _, cols in pre.transformers for c in cols]
    assert sorted(used) == sorted(feature_list), "preprocessor columns do not match the feature list"
    return clone(pre), source     # clone -> unfitted copy, refitted inside every pipeline

PREPROCESSORS = {}
for name, fname in [("FULL", "preprocessor_full.pkl"), ("HR", "preprocessor_hr.pkl")]:
    PREPROCESSORS[name], src = load_preprocessor(os.path.join(PREP_DIR, fname), FEATURE_LISTS[name])
    print(f"{name:<4} preprocessor: {src} ({len(FEATURE_LISTS[name])} input columns)")

## 2. Sanity checks before training

Before training anything we make sure the files are what we expect: right shapes, same class shares in train and test, no missing values, and no ID / leakage / target column hiding in X.

In [ ]:
print("Shapes:")
for n, o in [("X_train", X_train), ("X_test", X_test), ("y_train", y_train), ("y_test", y_test)]:
    print(f"  {n:<8} {o.shape}")
assert len(X_train) == len(y_train) == 76_804 and len(X_test) == len(y_test) == 19_202

dist = pd.concat([y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)],
                 axis=1, keys=["train_%", "test_%"]).mul(100).round(2).reindex(CLASS_ORDER)
dist["diff"] = (dist["train_%"] - dist["test_%"]).round(2)
display(dist)

n_missing = int(X_train.isna().sum().sum() + X_test.isna().sum().sum() + y_train.isna().sum() + y_test.isna().sum())
print("Missing values (X and y, train and test):", n_missing)
assert n_missing == 0

In [ ]:
forbidden = [ID_COL] + LEAKAGE_COLS + [TARGET_REG, TARGET_CLF]
for name, d in [("X_train", X_train), ("X_test", X_test)]:
    left = [c for c in forbidden if c in d.columns]
    assert not left, f"{name} contains forbidden columns: {left}"
    assert d.columns.tolist() == FEATURES_FULL, f"{name} columns are not FEATURES_FULL"
assert set(y_train.unique()) == set(y_test.unique()) == set(CLASS_ORDER)
assert set(FEATURES_HR) < set(FEATURES_FULL) and len(FEATURES_HR) == 22

print("OK: no ID, leakage or target column in X, and X columns == FEATURES_FULL (26).")
print("Personal columns not in HR:", [c for c in FEATURES_FULL if c not in FEATURES_HR])

## 3. Baseline

A model is only useful if it beats guessing. Two "dumb" baselines, evaluated with the **same 5-fold CV** as the real models:

- `most_frequent`: always predicts the biggest class (Severe).
- `stratified`: guesses randomly, following the class shares.

Every real model must beat both, otherwise it learned nothing.

All the evaluation helpers live here so every model is evaluated the same way. The pipeline (preprocessor + model) is passed to CV as one object, so **each fold fits its own scaler** on its own training part.

In [ ]:
CV_SCORING = {"f1_macro": "f1_macro", "accuracy": "accuracy", "f1_weighted": "f1_weighted"}

def make_pipeline(list_name, model):
    return Pipeline([("prep", clone(PREPROCESSORS[list_name])), ("model", clone(model))])

def cv_evaluate(pipe, X, y):
    res = cross_validate(pipe, X, y, cv=CV, scoring=CV_SCORING, n_jobs=-1)
    return {"cv_macro_f1_mean": res["test_f1_macro"].mean(), "cv_macro_f1_std": res["test_f1_macro"].std(),
            "cv_accuracy": res["test_accuracy"].mean(), "cv_weighted_f1": res["test_f1_weighted"].mean(),
            "fit_time_s": res["fit_time"].mean()}

cv_rows = []
for list_name, feats in FEATURE_LISTS.items():
    for strategy in ["most_frequent", "stratified"]:
        pipe = make_pipeline(list_name, DummyClassifier(strategy=strategy, random_state=RANDOM_STATE))
        cv_rows.append({"model": f"Dummy ({strategy})", "features": list_name, "best_params": "-",
                        **cv_evaluate(pipe, X_train[feats], y_train)})

baseline_table = pd.DataFrame(cv_rows)
display(baseline_table.round(4))
BASELINE_BEST_F1 = baseline_table["cv_macro_f1_mean"].max()

## 4. Models

Each model is trained with **both** feature lists:

- **FULL** (26 features): everything that is not leakage.
- **HR** (22 features): FULL without `sleep_hours_per_night`, `exercise_days_per_week`, `therapy_access`, `uses_therapy`. This is the realistic version, since an HR system doesn't know how you sleep or whether you go to therapy.

For each model and list we pass only that list's columns (`X_train[features]`). For HR this is the same as passing all 26 columns, because the HR preprocessor ignores the personal ones, but this way the saved HR pipeline really only needs the 22 HR columns.

**Tuning:** `RandomizedSearchCV`, 5-fold `StratifiedKFold(shuffle=True, random_state=42)`, `scoring="f1_macro"`, `n_iter=20`, `n_jobs=-1`. Only the training set is used.



1. the best parameters are evaluated again with 5-fold CV on the **full** training set (these are the CV numbers in the comparison table), and
2. the pipeline is refitted with those parameters on the **full** training set.

So all models are compared on the same thing: 5-fold CV on the full training set with the same folds. Logistic Regression is fast, so it is searched on the full training set directly.

In [ ]:
# stratified 30% subsample of the TRAINING set (only for the slow searches)
sub_idx, _ = train_test_split(X_train.index, train_size=SUBSAMPLE_FRAC,
                              stratify=y_train, random_state=RANDOM_STATE)
X_sub, y_sub = X_train.loc[sub_idx], y_train.loc[sub_idx]
print("Subsample for tuning:", X_sub.shape)
print((y_sub.value_counts(normalize=True) * 100).round(2).reindex(CLASS_ORDER).to_string())

fitted = {}        # (model, list) -> pipeline refitted on the full training set
encoded_y = {}     # (model, list) -> True if the model was trained on 0-3 labels (XGBoost)

def tune_model(model_name, estimator, param_dist, list_name, use_subsample=False, encode_y=False):
    feats = FEATURE_LISTS[list_name]
    pipe = make_pipeline(list_name, estimator)
    y_full = y_train.map(LABEL_TO_INT) if encode_y else y_train
    X_search, y_search = (X_sub[feats], y_sub) if use_subsample else (X_train[feats], y_train)
    if encode_y:
        y_search = y_search.map(LABEL_TO_INT)

    search = RandomizedSearchCV(pipe, param_dist, n_iter=N_ITER, scoring=CV_SCORING, refit=False,
                                cv=CV, n_jobs=-1, random_state=RANDOM_STATE)
    t0 = time.time()
    search.fit(X_search, y_search)
    search_time = time.time() - t0

    res = search.cv_results_
    best = int(np.argmin(res["rank_test_f1_macro"]))     # best by macro-F1
    best_params = res["params"][best]
    best_pipe = clone(pipe).set_params(**best_params)

    if use_subsample:   # re-evaluate the best params with 5-fold CV on the FULL training set
        scores = cv_evaluate(best_pipe, X_train[feats], y_full)
    else:               # the search already used the full training set
        scores = {"cv_macro_f1_mean": res["mean_test_f1_macro"][best], "cv_macro_f1_std": res["std_test_f1_macro"][best],
                  "cv_accuracy": res["mean_test_accuracy"][best], "cv_weighted_f1": res["mean_test_f1_weighted"][best],
                  "fit_time_s": res["mean_fit_time"][best]}

    if model_name in ("Random Forest", "XGBoost"):     # faster final fit (results don't change)
        best_pipe.set_params(model__n_jobs=-1)
    best_pipe.fit(X_train[feats], y_full)              # refit on the FULL training set
    fitted[(model_name, list_name)] = best_pipe
    encoded_y[(model_name, list_name)] = encode_y

    clean_params = {k.replace("model__", ""): (v.item() if hasattr(v, "item") else v) for k, v in best_params.items()}
    row = {"model": model_name, "features": list_name, "best_params": clean_params, **scores,
           "tuned_on": "30% subsample" if use_subsample else "full train", "search_time_s": search_time}
    cv_rows.append(row)
    print(f"{model_name:<22} {list_name:<4} CV macro-F1 = {scores['cv_macro_f1_mean']:.4f} "
          f"± {scores['cv_macro_f1_std']:.4f} | search {search_time/60:.1f} min | {clean_params}")
    return row

### 4.1 Logistic Regression

Multinomial logistic regression (with the default `lbfgs` solver, scikit-learn fits a multinomial (softmax) model when there are more than 2 classes; newer versions removed the `multi_class` argument, so we don't pass it). `max_iter=3000` so it converges.
Tuned: `C` (regularization strength) and `class_weight` (None vs "balanced").

In [ ]:
lr_params = {"model__C": loguniform(1e-3, 1e2),
             "model__class_weight": [None, "balanced"]}
for list_name in FEATURE_LISTS:
    tune_model("Logistic Regression", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
               lr_params, list_name)

### 4.2 Random Forest

Tuned: `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`, `class_weight`. Searched on the 30% subsample.

In [ ]:
rf_params = {"model__n_estimators": [100, 200],
             "model__max_depth": [None, 10, 20],
             "model__min_samples_leaf": [5, 10, 20],
             "model__max_features": ["sqrt", "log2", 0.3],
             "model__class_weight": [None, "balanced", "balanced_subsample"]}
for list_name in FEATURE_LISTS:
    tune_model("Random Forest", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
               rf_params, list_name, use_subsample=True)

### 4.3 HistGradientBoostingClassifier

Tuned: `learning_rate`, `max_iter`, `max_leaf_nodes`, `min_samples_leaf`, `l2_regularization`. Early stopping is turned off so `max_iter` really is the number of boosting rounds we tune. Searched on the 30% subsample.

In [ ]:
hgb_params = {"model__learning_rate": [0.02, 0.05, 0.1, 0.2],
              "model__max_iter": [100, 200, 300],
              "model__max_leaf_nodes": [15, 31, 63],
              "model__min_samples_leaf": [20, 50, 100, 200],
              "model__l2_regularization": [0.0, 0.1, 1.0, 10.0]}
for list_name in FEATURE_LISTS:
    tune_model("HistGradientBoosting",
               HistGradientBoostingClassifier(early_stopping=False, random_state=RANDOM_STATE),
               hgb_params, list_name, use_subsample=True)

### 4.4 Optional: XGBoost

XGBoost needs numeric labels, so the target is encoded with the **ordinal** mapping Low 0, Moderate 1, High 2, Severe 3, and predictions are decoded back to names for every report.

In [ ]:
if HAS_XGB:
    xgb_params = {"model__n_estimators": [200, 300, 500],
                  "model__learning_rate": [0.03, 0.05, 0.1],
                  "model__max_depth": [3, 4, 6],
                  "model__min_child_weight": [1, 5, 10],
                  "model__subsample": [0.8, 1.0],
                  "model__colsample_bytree": [0.7, 1.0],
                  "model__reg_lambda": [1.0, 5.0]}
    for list_name in FEATURE_LISTS:
        tune_model("XGBoost",
                   XGBClassifier(objective="multi:softprob", tree_method="hist", eval_metric="mlogloss",
                                 random_state=RANDOM_STATE, n_jobs=1),
                   xgb_params, list_name, use_subsample=True, encode_y=True)
else:
    print("XGBoost is not installed -> skipped (optional).")

## 5. Metrics

**Main metric: macro-F1.** It averages the F1 of the 4 classes equally, so the smallest class (High, 19%) counts as much as the others. We also report:

- **Accuracy** and **weighted F1**.
- **Per-class precision / recall / F1** and the **confusion matrix** (counts and row-normalized), always in the order Low, Moderate, High, Severe.
- **Quadratic Weighted Kappa (QWK)** on the 0–3 encoded labels. Plain accuracy treats every mistake the same, but predicting Severe for a Low employee is much worse than predicting Moderate. QWK punishes far mistakes more.
- **Far-off error rate:** % of predictions that are 2 or more classes away from the truth (for example Low predicted as High or Severe). These are the mistakes that would really mislead an HR manager.

The helpers below also decode XGBoost's 0–3 output back to class names and always return probabilities in the ordinal order.

In [ ]:
def to_labels(pred):
    pred = np.asarray(pred)
    if pred.dtype.kind in "iuf":        # XGBoost returns 0-3
        return np.array([INT_TO_LABEL[int(p)] for p in pred])
    return pred.astype(str)

def predict_labels(pipe, X):
    return to_labels(pipe.predict(X))

def proba_ordinal(pipe, X):
    # predict_proba columns re-ordered to Low, Moderate, High, Severe
    proba = pipe.predict_proba(X)
    classes = list(to_labels(pipe.classes_))
    return pd.DataFrame(proba, columns=classes, index=getattr(X, "index", None))[CLASS_ORDER]

def far_off_rate(y_true, y_pred):
    t = np.array([LABEL_TO_INT[v] for v in y_true]); p = np.array([LABEL_TO_INT[v] for v in y_pred])
    return np.mean(np.abs(t - p) >= 2) * 100

def evaluate(y_true, y_pred):
    t = np.array([LABEL_TO_INT[v] for v in y_true]); p = np.array([LABEL_TO_INT[v] for v in y_pred])
    return {"macro_f1": f1_score(y_true, y_pred, average="macro", labels=CLASS_ORDER),
            "accuracy": accuracy_score(y_true, y_pred),
            "weighted_f1": f1_score(y_true, y_pred, average="weighted", labels=CLASS_ORDER),
            "qwk": cohen_kappa_score(t, p, weights="quadratic"),
            "far_off_error_%": far_off_rate(y_true, y_pred)}

def plot_confusion(y_true, y_pred, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for ax, norm, sub in [(axes[0], None, "counts"), (axes[1], "true", "row-normalized (recall per class)")]:
        cm = confusion_matrix(y_true, y_pred, labels=CLASS_ORDER, normalize=norm)
        ConfusionMatrixDisplay(cm, display_labels=CLASS_ORDER).plot(
            ax=ax, cmap="Blues", colorbar=False, values_format=".2f" if norm else "d")
        ax.set_title(sub, fontsize=11)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    fig.suptitle(title, fontsize=13)
    plt.tight_layout(); plt.show()

## 6. Model selection and final test evaluation

### 6.1 CV comparison table (training set only)

This table is the **only** thing used to choose the model. The test set is not touched yet.

In [ ]:
cv_table = pd.DataFrame(cv_rows)
show_cols = ["model", "features", "cv_macro_f1_mean", "cv_macro_f1_std", "cv_accuracy", "cv_weighted_f1",
             "fit_time_s", "tuned_on", "best_params"]
cv_table_view = cv_table.reindex(columns=show_cols).sort_values(["features", "cv_macro_f1_mean"], ascending=[True, False])
display(cv_table_view.round(4).reset_index(drop=True))

model_rows = cv_table[~cv_table["model"].str.startswith("Dummy")]
assert (model_rows["cv_macro_f1_mean"] > BASELINE_BEST_F1).all(), "a model did not beat the baselines!"
print(f"All tuned models beat the best baseline CV macro-F1 ({BASELINE_BEST_F1:.4f}).")

### 6.2 Choosing the final model

Rule: highest **CV macro-F1 on the FULL list**. If another model is within about **0.01** of the best, we prefer the simpler / more explainable one (Logistic Regression < Random Forest < HistGradientBoosting < XGBoost). The same model type is then used for both FULL and HR, so the two versions can be compared fairly and the backend only deals with one kind of model.

In [ ]:
SIMPLICITY = {"Logistic Regression": 0, "Random Forest": 1, "HistGradientBoosting": 2, "XGBoost": 3}
TOLERANCE = 0.01

full_rank = model_rows[model_rows["features"] == "FULL"].sort_values("cv_macro_f1_mean", ascending=False)
best_f1 = full_rank["cv_macro_f1_mean"].iloc[0]
candidates = full_rank[full_rank["cv_macro_f1_mean"] >= best_f1 - TOLERANCE].copy()
candidates["simplicity"] = candidates["model"].map(SIMPLICITY)
CHOSEN_MODEL = candidates.sort_values(["simplicity", "cv_macro_f1_mean"], ascending=[True, False])["model"].iloc[0]

print("CV macro-F1 ranking (FULL):")
print(full_rank[["model", "cv_macro_f1_mean", "cv_macro_f1_std"]].round(4).to_string(index=False))
print("\nCV macro-F1 ranking (HR):")
print(model_rows[model_rows["features"] == "HR"].sort_values("cv_macro_f1_mean", ascending=False)
      [["model", "cv_macro_f1_mean", "cv_macro_f1_std"]].round(4).to_string(index=False))
print(f"\nModels within {TOLERANCE} of the best: {candidates['model'].tolist()}")
print(f"CHOSEN MODEL: {CHOSEN_MODEL}  (chosen with CV on the training set only)")

### 6.3 Test set results (reported for transparency)

The choice above is already final. Here we look at the test set for the first time and report every tuned model.

In [ ]:
test_rows, test_preds = [], {}
for (model_name, list_name), pipe in fitted.items():
    pred = predict_labels(pipe, X_test[FEATURE_LISTS[list_name]])
    test_preds[(model_name, list_name)] = pred
    test_rows.append({"model": model_name, "features": list_name, **evaluate(y_test, pred)})

test_table = pd.DataFrame(test_rows).sort_values(["features", "macro_f1"], ascending=[True, False]).reset_index(drop=True)
display(test_table.round(4))
print(f"Reminder: the final model ({CHOSEN_MODEL}) was chosen with CV, not with this table.")

### 6.4 FULL vs HR for the chosen model

How much do we lose when the model can't use the 4 personal/lifestyle columns (sleep, exercise, therapy access, uses therapy)?

In [ ]:
fh = test_table[test_table["model"] == CHOSEN_MODEL].set_index("features").loc[["FULL", "HR"]]
cv_fh = cv_table[cv_table["model"] == CHOSEN_MODEL].set_index("features").loc[["FULL", "HR"]]
full_vs_hr = pd.DataFrame({"CV macro-F1": cv_fh["cv_macro_f1_mean"],
                           "test macro-F1": fh["macro_f1"], "test accuracy": fh["accuracy"],
                           "test QWK": fh["qwk"], "test far-off %": fh["far_off_error_%"]})
full_vs_hr.loc["HR - FULL"] = full_vs_hr.loc["HR"] - full_vs_hr.loc["FULL"]
display(full_vs_hr.round(4))

### 6.5 Chosen model: confusion matrix and classification report (test set)

In [ ]:
for list_name in ["FULL", "HR"]:
    pred = test_preds[(CHOSEN_MODEL, list_name)]
    print(f"===== {CHOSEN_MODEL} | {list_name} | test set =====")
    print(classification_report(y_test, pred, labels=CLASS_ORDER, digits=3))
    plot_confusion(y_test, pred, f"{CHOSEN_MODEL} ({list_name}) – test set")

**How to read it:** the row-normalized matrix shows, for each true level, where the employees ended up. Most mistakes should sit next to the diagonal (for example High predicted as Moderate or Severe). Those are "near misses". The far-off error rate in the tables above measures the dangerous ones (2+ levels away).

## 7. Explainability

Bayyinah promises explainable predictions, so we need to know which inputs drive the model.

**Permutation importance:** shuffle one original column in the test set and see how much macro-F1 drops. A big drop means the model relies on that column. Because we shuffle the **raw** columns and run the **whole pipeline**, we get one number per original feature (e.g. `job_role`), not one per one-hot column. `n_repeats=5`, `scoring="f1_macro"`.

In [ ]:
def perm_importance(pipe, list_name, encode):
    feats = FEATURE_LISTS[list_name]
    y_eval = y_test.map(LABEL_TO_INT) if encode else y_test
    r = permutation_importance(pipe, X_test[feats], y_eval, scoring="f1_macro",
                               n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
    return (pd.DataFrame({"feature": feats, "importance_mean": r.importances_mean, "importance_std": r.importances_std})
              .sort_values("importance_mean", ascending=False).reset_index(drop=True))

def plot_importance(imp, title):
    d = imp.sort_values("importance_mean")
    fig, ax = plt.subplots(figsize=(8, 0.32 * len(d) + 1.2))
    ax.barh(d["feature"], d["importance_mean"], xerr=d["importance_std"], color="#2a78d6",
            height=0.6, error_kw={"ecolor": "#6b6b6b", "lw": 1})
    ax.axvline(0, color="#6b6b6b", lw=0.8)
    ax.set_xlabel("Drop in macro-F1 when the column is shuffled")
    ax.set_title(title)
    ax.grid(axis="x", color="#e6e6e6", lw=0.6); ax.set_axisbelow(True)
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    plt.tight_layout(); plt.show()

importances = {}
for list_name in ["FULL", "HR"]:
    key = (CHOSEN_MODEL, list_name)
    importances[list_name] = perm_importance(fitted[key], list_name, encoded_y[key])
    print(f"Top 8 features – {CHOSEN_MODEL} ({list_name})")
    print(importances[list_name].head(8).round(4).to_string(index=False), "\n")
    plot_importance(importances[list_name], f"Permutation importance – {CHOSEN_MODEL} ({list_name}, test set)")

### 7.2 What drives the prediction?

*(Numbers are from our run. If your re-run gives slightly different values, use the printed tables above.)*

- **Workload is the strongest driver:** `work_hours_per_week` and `meetings_per_day` are the top two in both FULL and HR. This makes sense, since long hours and many meetings are classic burnout causes.
- **Recovery matters:** `sleep_hours_per_night` is almost as important as meetings in FULL, and `vacation_days_taken` and `exercise_days_per_week` are also high. These are about rest and recovery, which also makes sense.
- **Support at work:** `social_support_score`, `manager_support_score`, `deadline_pressure_score` and `autonomy_score` come next, with smaller effects.
- **Almost no effect:** `job_role`, `industry`, `country`, `salary_usd`, `age`, `company_size`, and surprisingly `work_life_balance_score` and `job_satisfaction_score` (about 0). In real life we would expect those two to matter. Here it probably just means the data generator did not use them to create burnout. `seniority_level` has only a small importance even though Lead/Manager had much higher burnout in the original data. Part of that pattern may be carried by other columns, and the title fixes weakened it a little (we did not test which).
- **Why HR loses accuracy:** in FULL, sleep and exercise are among the top 6 features, and the HR model simply does not have them. That explains most of the FULL → HR drop.

**Important:** the data is synthetic, so these importances describe **how the dataset was generated**, not proven real-world causes of burnout. Also, permutation importance can split credit between correlated columns, so a small value does not always mean the column is useless.

## 9. Save for the backend

We save the final pipelines (preprocessor + model, already refitted on the full training set) for the chosen model with both feature lists, the result tables, and a model card with everything needed to reproduce it.

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)
FINAL = {"FULL": fitted[(CHOSEN_MODEL, "FULL")], "HR": fitted[(CHOSEN_MODEL, "HR")]}
PATHS = {"FULL": os.path.join(MODELS_DIR, "burnout_classifier_full.pkl"),
         "HR": os.path.join(MODELS_DIR, "burnout_classifier_hr.pkl")}
for list_name, pipe in FINAL.items():
    joblib.dump(pipe, PATHS[list_name])

# results table: CV rows + test rows in one CSV
cv_out = cv_table.reindex(columns=show_cols).assign(evaluated_on="5-fold CV (train)")
cv_out["best_params"] = cv_out["best_params"].astype(str)
test_out = test_table.assign(evaluated_on="test set")
results = pd.concat([cv_out, test_out], ignore_index=True)
results.to_csv(os.path.join(MODELS_DIR, "classification_results.csv"), index=False)

def r4(x): return round(float(x), 4)
model_card = {
    "task": "classification of burnout_level",
    "chosen_model": CHOSEN_MODEL,
    "selection_rule": f"highest 5-fold CV macro-F1 on FULL; simpler model preferred within {TOLERANCE}",
    "class_order": CLASS_ORDER,
    "label_encoding": LABEL_TO_INT,
    "versions": {},
    "random_state": RANDOM_STATE,
    "date": datetime.date.today().isoformat(),
    "data": "synthetic (mental_health_burnout_tech_2026), consistency-corrected, leakage removed; "
            "train 76,804 / test 19,202 (80/20 stratified, random_state=42)",
    "tuning": f"RandomizedSearchCV, StratifiedKFold(5, shuffle=True, random_state=42), scoring=f1_macro, n_iter={N_ITER}; "
              "tree models searched on a stratified 30% subsample of train, then CV-evaluated and refitted on full train",
    "models": {},
}
for list_name in ["FULL", "HR"]:
    cvr = cv_table[(cv_table["model"] == CHOSEN_MODEL) & (cv_table["features"] == list_name)].iloc[0]
    tr = test_table[(test_table["model"] == CHOSEN_MODEL) & (test_table["features"] == list_name)].iloc[0]
    model_card["models"][list_name] = {
        "file": os.path.basename(PATHS[list_name]),
        "features": FEATURE_LISTS[list_name],
        "best_params": cvr["best_params"],
        "cv": {"macro_f1_mean": r4(cvr["cv_macro_f1_mean"]), "macro_f1_std": r4(cvr["cv_macro_f1_std"]),
               "accuracy": r4(cvr["cv_accuracy"]), "weighted_f1": r4(cvr["cv_weighted_f1"])},
        "test": {k: r4(tr[k]) for k in ["macro_f1", "accuracy", "weighted_f1", "qwk", "far_off_error_%"]},
    }
model_card["versions"] = {"pandas": pd.__version__, "scikit-learn": sklearn.__version__,
                          "numpy": np.__version__, "joblib": joblib.__version__}
if CHOSEN_MODEL == "XGBoost":
    model_card["versions"]["xgboost"] = xgboost.__version__

with open(os.path.join(MODELS_DIR, "model_card.json"), "w") as f:
    json.dump(model_card, f, indent=2, default=str)

for name in sorted(os.listdir(MODELS_DIR)):
    print(f"{name:<32} {os.path.getsize(os.path.join(MODELS_DIR, name)) / 1024:>10,.1f} KB")

### 9.1 `predict_burnout`: the function the backend calls

It takes one employee as a dict of **raw** features (missing keys or `None` are fine, the imputers fill them), and returns the predicted level plus the probability of each level in the order Low, Moderate, High, Severe.

An unseen category (for example a new job role) is encoded as all zeros by the `OneHotEncoder(handle_unknown="ignore")`, so the prediction simply relies on the other columns. It does not crash

In [ ]:
def predict_burnout(employee: dict, pipeline) -> dict:
    cols = list(pipeline.feature_names_in_)                     # FULL or HR columns, depending on the pipeline
    row = pd.DataFrame([{c: employee.get(c, np.nan) for c in cols}], columns=cols)
    row = row.replace({None: np.nan})
    for c in cols:                                              # numeric columns must be numbers (or NaN)
        if c in FL["numeric"]:
            row[c] = pd.to_numeric(row[c], errors="coerce")
    proba = proba_ordinal(pipeline, row).iloc[0]
    return {"predicted_level": str(predict_labels(pipeline, row)[0]),
            "probabilities": {c: round(float(proba[c]), 4) for c in CLASS_ORDER}}

# example 1: a real row from X_test
example = X_test.iloc[0].to_dict()
print("True level:", y_test.iloc[0])
print("FULL:", predict_burnout(example, FINAL["FULL"]))
print("HR  :", predict_burnout(example, FINAL["HR"]))

In [ ]:
# example 2: some missing values + a job role the model has never seen
messy = X_test.iloc[1].to_dict()
messy["salary_usd"] = None
messy["sleep_hours_per_night"] = np.nan     # only used by FULL
messy["work_mode"] = None
messy["job_role"] = "Prompt Engineer"       # unseen category
del messy["team_size"]                      # key missing completely

print("FULL:", predict_burnout(messy, FINAL["FULL"]))
print("HR  :", predict_burnout(messy, FINAL["HR"]))

## 10. Final checks

In [ ]:
# 1) reloaded .pkl files must give exactly the same predictions and probabilities
for list_name in ["FULL", "HR"]:
    feats = FEATURE_LISTS[list_name]
    reloaded = joblib.load(PATHS[list_name])
    same_pred = np.array_equal(FINAL[list_name].predict(X_test[feats]), reloaded.predict(X_test[feats]))
    same_proba = np.array_equal(FINAL[list_name].predict_proba(X_test[feats]), reloaded.predict_proba(X_test[feats]))
    assert same_pred and same_proba
    print(f"{list_name:<4} reloaded pipeline: identical predictions = {same_pred}, identical probabilities = {same_proba}")

**2) The test set was not used for any decision.**

- All hyperparameter searches used only `X_train` / `y_train` (or a subsample of it) with 5-fold stratified CV.
- The final model was chosen in section 6.2 from the CV table, **before** any test prediction was made (section 6.3).
- No threshold, class weight or setting was changed after looking at the test results.
- The test set was only used to report results (6.3–6.5), for permutation importance (7)

In [ ]:
# 3) versions for the backend requirements.txt
print("pandas       :", pd.__version__)
print("scikit-learn :", sklearn.__version__)
print("numpy        :", np.__version__)
print("joblib       :", joblib.__version__)
req = f"pandas=={pd.__version__}\nscikit-learn=={sklearn.__version__}\nnumpy=={np.__version__}\njoblib=={joblib.__version__}"
if CHOSEN_MODEL == "XGBoost":
    req += f"\nxgboost=={xgboost.__version__}"
print("\nrequirements.txt:\n" + req)

Optional: download the `models/` folder as one zip.

In [ ]:
shutil.make_archive("bayyinah_models", "zip", MODELS_DIR)
try:
    from google.colab import files
    files.download("bayyinah_models.zip")
except ImportError:
    print("Not in Colab – files are in the 'models/' folder and bayyinah_models.zip.")